# E14 — training `e14_q`: `e12_final`'s recipe with the query weight

Registration: `research/e14/PREREG_E14.md` §2.1. Everything is `e12_final`'s: the same
start, data, seed, batches, 20,000 steps, objective and pinned loss constant. **One
change:** every training step inside a query counts **4×** in the loss. Every other
step counts exactly what it did, and validation is unweighted.

**Before any training, cell 3 rebuilds E11's start and checks it**, and checks that a
384-token step fits. If either fails it stops before a session is spent. **Cell 5
reads the run back** against `e12_final`'s record. It must print
`MISMATCHES: none -- e14_q is as registered`.

**Attach two inputs** (right panel → *Add Input*):
1. **`uie-corpus2`**, the data bundle E7–E13 used;
2. **`uie-e14-code`**.

**Settings:** GPU T4, Internet **On**. **Save Version → Save & Run All.** About 1.5 h.
**A rerun skips a finished arm.** When done, download `/kaggle/working`.

In [ ]:
# --- 1. locate the code bundle -----------------------------------------------------
import hashlib, json
from pathlib import Path


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


root = Path("/kaggle/input")
code_hits = sorted(root.rglob("E14_CODE_MANIFEST.json")) if root.exists() else []
if not code_hits:
    raise SystemExit("Attach the uie-e14-code dataset (right panel -> Add Input), then rerun.")
CODE = code_hits[0].parent
WORK = Path("/kaggle/working")
print("code:", CODE)

data_hits = sorted(root.rglob("bundle.manifest.json")) if root.exists() else []
if not data_hits:
    raise SystemExit("Attach the uie-corpus2 dataset too, then rerun.")
DATA = data_hits[0].parent
print("data:", DATA)

In [ ]:
# --- 2. environment: install first, then run THIS code and prove it ------------
import importlib, subprocess, sys
from importlib import metadata

TLDEXTRACT = "5.3.1"          # the version the local evaluation parses with


def installed(dist):
    try:
        return metadata.version(dist)
    except metadata.PackageNotFoundError:
        return None


if installed("tldextract") != TLDEXTRACT:
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        f"tldextract=={TLDEXTRACT}"], check=True)
    except subprocess.CalledProcessError:
        raise SystemExit("pip could not install tldextract: Settings -> Internet -> On, rerun.")
    importlib.invalidate_caches()
assert installed("tldextract") == TLDEXTRACT, installed("tldextract")

# CODE ONLY on the path: the data bundle carries an older `uie`.
for p in (CODE / "research" / "e14", CODE / "research" / "e13", CODE / "research" / "native", CODE):
    sys.path.insert(0, str(p))
for name in [m for m in sys.modules if m == "uie" or m.startswith("uie.")
             or m in ("e8_common", "e11_common", "e12_common", "e13_common", "e14_common")]:
    del sys.modules[name]
import uie
import e12_common as K
import e14_common as E14
manifest = json.loads((CODE / "E14_CODE_MANIFEST.json").read_text())
for mod in (uie, K, E14):
    assert str(CODE) in mod.__file__, f"{mod.__name__} is SHADOWED -- check sys.path"
bad = [f for f, h in manifest["files"].items() if sha256(CODE / f) != h]
bad += [t for t, h in manifest["tables"].items() if sha256(CODE / t) != h]
if sha256(CODE / "e12_final.pt") != manifest["e12_final"]:
    bad.append("e12_final.pt")
assert not bad, f"STOP. files differ from the manifest: {bad}"
print(f"code: commit {manifest['commit'][:12]}; {len(manifest['files'])} files, "
      f"{len(manifest['tables'])} tables and e12_final.pt verified")

import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Settings -> Accelerator -> GPU T4, then rerun.")
GPU = torch.cuda.get_device_name(0)
print("torch:", torch.__version__, "| gpu:", GPU)

need = ["name_train", "structure_train", "phishing_name_train", "phishing_structure_train",
        "name_val", "structure_val", "phishing_name_val", "phishing_structure_val"]
present = {p.name for p in DATA.iterdir()}
missing = [s for s in need if f"{s}.jsonl" not in present and f"{s}.jsonl.gz" not in present]
if missing:
    raise SystemExit(f"STOP. missing {missing}: this is not the uie-corpus2 bundle.")
import inspect
from uie.kaggle_train import run
assert {"query_weight", "slash_plan", "max_len", "e12_start", "loss_constant"} <= set(
    inspect.signature(run).parameters)
print("data bundle: all 8 streams present; E14 option present")
print("e14_q:", E14.recipe_q())

In [ ]:
# --- 3. THE START CHECK and the memory preflight, before any training -----------------
e10, e11 = K.start_fingerprints()
print("E10 start:", e10, "(recorded", K.E10_START + ")")
print("E11 start:", e11, "(recorded", K.E11_START + ")")
if (e10, e11) != (K.E10_START, K.E11_START):
    raise SystemExit("STOP. This torch rebuilds a different start than E10/E11 recorded. "
                     "Nothing was trained.")
try:
    peak = K.memory_preflight(torch.device("cuda"))
except torch.cuda.OutOfMemoryError:
    raise SystemExit("STOP. A full 256 x 384 training step does not fit on this GPU.")
total = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"start check PASSED; memory preflight PASSED ({peak:.1f} of {total:.1f} GB)")

In [ ]:
# --- 4. train e14_q (skipped if finished) -------------------------------------------
import time

PROV = WORK / "PROVENANCE.json"
prov = json.loads(PROV.read_text()) if PROV.exists() else {"arms": {}}
prov.update({"registration": E14.REGISTRATION, "code_commit": manifest["commit"],
             "start_check": {"e10": e10, "e11": e11}, "gpu": GPU})
out = WORK / "e14_q"
if (out / "train_result.json").exists() and (out / "final.pt").exists():
    print("e14_q: finished earlier -- skipped")
else:
    out.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    run(DATA, out, **E14.recipe_q())
    prov["arms"]["e14_q"] = {"wall_minutes": round((time.time() - t0) / 60, 1)}
    PROV.write_text(json.dumps(prov, indent=1))

In [ ]:
# --- 5. read e14_q back against e12_final's record (V5) -------------------------------
problems = []
ck = torch.load(WORK / "e14_q" / "final.pt", map_location="cpu", weights_only=False)
e12 = torch.load(CODE / "e12_final.pt", map_location="cpu", weights_only=False)
tc, mc, tk = ck["train_config"], ck["model_config"], ck["tokenizer"]
r = json.loads((WORK / "e14_q" / "train_result.json").read_text())
s = r.get("e12_start") or {}
q = r.get("query_weight") or {}
want = [(f"train_config {k}", tc.get(k), v) for k, v in e12["train_config"].items()]
want += [(f"model_config {k}", mc.get(k), v) for k, v in e12["model_config"].items()]
want += [("query_weight", tc.get("query_weight"), E14.QUERY_WEIGHT),
         ("extra settings", sorted(set(tc) - set(e12["train_config"])), ["query_weight"]),
         ("tokenizer", tk == e12["tokenizer"], True),
         ("init weights", r.get("init_weights_sha256"), E14.E12_FINAL_INIT),
         ("row counts", r.get("counts"), E14.E12_FINAL_COUNTS),
         ("dropped over max_len", r.get("dropped_over_max_len"), E14.E12_FINAL_DROPPED),
         ("loss constant", r.get("loss_constant"), E14.E12_FINAL_LOSS_CONSTANT),
         ("band steps (same batches)",
          {k: v["steps"] for k, v in (r.get("grad_diagnostics") or {}).items() if k.startswith("all ")},
          E14.E12_FINAL_BAND_STEPS),
         ("query-weight record", q.get("w"), E14.QUERY_WEIGHT)]
want += [(f"start: {k}", s.get(k), v) for k, v in E14.E12_FINAL_START.items()]
problems = [w for w in want if w[1] != w[2]]
prov["arms"].setdefault("e14_q", {})
prov["arms"]["e14_q"].update({"best_sha256": sha256(WORK / "e14_q" / "best.pt"),
                              "final_sha256": sha256(WORK / "e14_q" / "final.pt"),
                              "init_weights_sha256": r.get("init_weights_sha256"),
                              "query_weight": q, "final_val": r.get("final"),
                              "counts": r.get("counts")})
PROV.write_text(json.dumps(prov, indent=1))
print(f"e14_q val {r.get('final')}; query weight record {q}")
print("MISMATCHES:", problems if problems else "none -- e14_q is as registered")
print("\nNow download /kaggle/working (e14_q/, PROVENANCE.json).")